# NPDES Data Cleaning: Catchments (Iowa)

Cleans the national NPDES → NHDPlus catchment crosswalk down to Iowa, giving one
tidy row per facility **outfall** (`npdes_id`, `sub_id`) located in its receiving
catchment / HUC-12. This links each permitted discharge point to the watershed it
drains into.

**Input:**  `data/tabular/01_raw/npdes/NPDES_CATCHMENTS.csv` (national)
**Output:** `data/tabular/02_clean/npdes/npdes-catchments-clean.csv`

| raw column          | meaning                                   | type    |
|---------------------|-------------------------------------------|---------|
| `NPDES_ID`          | NPDES permit number (join key)            | id      |
| `SUB_ID`            | outfall / sub-facility id                 | id      |
| `PERMIT_TYPE_CODE`  | permit type (NPD, UFT, …)                 | code    |
| `LATITUDE83`/`LONGITUDE83` | outfall coordinates (NAD83)        | deg     |
| `NHDPLUSID`         | NHDPlus catchment id                      | id      |
| `WBD_HU12`          | HUC-12 watershed code                     | id      |
| `CATCHMENT_HUC12`   | catchment's HUC-12 code                   | id      |
| `REACHCODE`/`GNIS_NAME` | receiving stream reach / name         | id/text |
| `AREASQKM`/`LENGTHKM`   | catchment area / flowline length      | km      |
| `NAVIGABLE`/`HEADWATER`/`COASTAL`/`TIDAL`/`ALASKAN` | `Y`/`N` flags | bool |

**Cleaning steps** — filter to Iowa, type the coordinates and measures,
zero-pad the HUC-12 / NHDPlus ids back to strings, convert the `Y`/`N` flags to
booleans, then de-duplicate on (`npdes_id`, `sub_id`).

In [1]:
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "npdes"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "npdes"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Project scope: this is an Iowa water-quality study, so the national crosswalk
# is narrowed to Iowa (STATECODE == "IA") after the schema guards run.
STATE = "IA"
print("Repo root:", REPO_ROOT)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/npdes


## Step 1 — Load (national) and filter to Iowa

Everything is read as a string so the NHDPlus / HUC identifiers keep their exact
digits; numeric coercion is explicit below.

In [2]:
df = pd.read_csv(RAW_DIR / "NPDES_CATCHMENTS.csv", dtype="string")
n_national = len(df)
print(f"Loaded {n_national:,} national catchment rows")
print("Columns:", list(df.columns))

df = df[df["STATECODE"] == STATE].copy()
print(f"\nFiltered to {STATE}: {len(df):,} rows, {df['NPDES_ID'].nunique()} permits")
print("\nNulls per column (Iowa):")
print(df.isna().sum().to_string())

Loaded 1,255,164 national catchment rows
Columns: ['NPDES_ID', 'PERMIT_TYPE_CODE', 'PERMIT_TYPE_DESC', 'SUB_ID', 'LATITUDE83', 'LONGITUDE83', 'STATECODE', 'NHDPLUSID', 'WBD_HU12', 'WBD_HU12NAME', 'REACHCODE', 'GNIS_NAME', 'CATCHMENT_HUC12', 'AREASQKM', 'LENGTHKM', 'NAVIGABLE', 'HEADWATER', 'COASTAL', 'TIDAL', 'ALASKAN']

Filtered to IA: 2,432 rows, 2156 permits

Nulls per column (Iowa):
NPDES_ID               0
PERMIT_TYPE_CODE       0
PERMIT_TYPE_DESC       0
SUB_ID                 0
LATITUDE83             0
LONGITUDE83            0
STATECODE              0
NHDPLUSID              0
WBD_HU12               0
WBD_HU12NAME           0
REACHCODE              0
GNIS_NAME           1022
CATCHMENT_HUC12        0
AREASQKM               0
LENGTHKM               0
NAVIGABLE              0
HEADWATER              0
COASTAL                0
TIDAL                  0
ALASKAN                0


## Step 2 — Rename and select columns

Lower-case to snake_case and keep the geographic + watershed fields. `STATECODE`
is dropped now that it is constant.

In [3]:
RENAME = {
    "NPDES_ID": "npdes_id", "SUB_ID": "sub_id",
    "PERMIT_TYPE_CODE": "permit_type_code", "PERMIT_TYPE_DESC": "permit_type_desc",
    "LATITUDE83": "latitude", "LONGITUDE83": "longitude",
    "NHDPLUSID": "nhdplusid", "WBD_HU12": "wbd_huc12", "WBD_HU12NAME": "wbd_huc12_name",
    "REACHCODE": "reachcode", "GNIS_NAME": "gnis_name",
    "CATCHMENT_HUC12": "catchment_huc12", "AREASQKM": "area_sqkm", "LENGTHKM": "length_km",
    "NAVIGABLE": "navigable", "HEADWATER": "headwater",
    "COASTAL": "coastal", "TIDAL": "tidal", "ALASKAN": "alaskan",
}
df = df.rename(columns=RENAME)[list(RENAME.values())]
for c in ["npdes_id", "sub_id", "permit_type_code", "permit_type_desc",
          "gnis_name", "wbd_huc12_name"]:
    df[c] = df[c].str.strip()
df.head(2)

,npdes_id,sub_id,permit_type_code,permit_type_desc,latitude,longitude,nhdplusid,wbd_huc12,wbd_huc12_name,reachcode,gnis_name,catchment_huc12,area_sqkm,length_km,navigable,headwater,coastal,tidal,alaskan
491,IA0000787,001,NPD,NPDES Individual Permit,40.83224,-91.10224,6964563,070801041204,Flint Creek,07080104002443,Flint Creek,070801041204,12.0419999794503,9.58352320300944,Y,N,N,N,N
835,IAU001298,BESTPICK,UFT,Unpermitted Facility,42.483393,-96.387161,17243984,102300010305,Bacon Creek-Missouri River,10230001000534,<NA>,102300010305,20.9286000161712,1.74439576072448,Y,Y,N,N,N


## Step 3 — Type coordinates and measures

Coordinates, catchment area, and flowline length are quantities; coerce to
float and confirm the Iowa bounding box.

In [4]:
for c in ["latitude", "longitude", "area_sqkm", "length_km"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

coord = df.dropna(subset=["latitude", "longitude"])
assert coord["latitude"].between(40, 44).all(), "latitude outside Iowa bbox"
assert coord["longitude"].between(-97, -90).all(), "longitude outside Iowa bbox"
print("lat range:", coord["latitude"].min(), "→", coord["latitude"].max())
print("lon range:", coord["longitude"].min(), "→", coord["longitude"].max())
print(df[["area_sqkm", "length_km"]].describe().to_string())

lat range: 40.38601 → 43.49822
lon range: -96.61722 → -90.1695
       area_sqkm  length_km
count     2432.0     2432.0
mean     8.22328   3.409553
std     14.30235   2.726482
min        0.009   0.061278
25%      2.15505   1.711658
50%      4.24755   2.750394
75%       8.8524   4.234131
max      237.735   25.50484


## Step 4 — Normalize HUC-12 identifiers

A HUC-12 is a 12-digit code and a HUC-8 (`reachcode` prefix etc.) must keep
leading zeros, so re-pad any that lost a leading zero on the round-trip through
the raw export. Validate the two HUC-12 columns are 12 digits.

In [5]:
for c in ["wbd_huc12", "catchment_huc12"]:
    df[c] = df[c].str.replace(r"\.0$", "", regex=True).str.zfill(12)
    assert df[c].dropna().str.fullmatch(r"\d{12}").all(), f"{c} not 12 digits"
print(df[["wbd_huc12", "catchment_huc12"]].head(3).to_string())

         wbd_huc12 catchment_huc12
491   070801041204    070801041204
835   102300010305    102300010305
2916  071000061401    071000061401


## Step 5 — Boolean flags

`NAVIGABLE`, `HEADWATER`, `COASTAL`, `TIDAL`, and `ALASKAN` are `Y`/`N`
indicators. Convert to booleans after confirming the vocabulary.

In [6]:
FLAG_COLS = ["navigable", "headwater", "coastal", "tidal", "alaskan"]
for c in FLAG_COLS:
    vals = set(df[c].dropna().unique())
    assert vals <= {"Y", "N"}, f"{c}: unexpected values {vals}"
    df[c] = df[c].eq("Y")
print(df[FLAG_COLS].sum().to_string())

navigable    2432
headwater     799
coastal         0
tidal           0
alaskan         0


## Step 6 — De-duplicate, sort, and write

(`npdes_id`, `sub_id`) uniquely identifies an outfall. Drop exact duplicates,
confirm the key, sort, and write.

In [7]:
df = df.drop_duplicates().sort_values(["npdes_id", "sub_id"]).reset_index(drop=True)
dupes = df.duplicated(["npdes_id", "sub_id"]).sum()
assert dupes == 0, f"{dupes} duplicate (npdes_id, sub_id) rows"

out_path = CLEAN_DIR / "npdes-catchments-clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df):,} rows × {df.shape[1]} cols (from {n_national:,} national) to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Wrote 2,432 rows × 19 cols (from 1,255,164 national) to:
  data/tabular/02_clean/npdes/npdes-catchments-clean.csv


,npdes_id,sub_id,permit_type_code,permit_type_desc,latitude,longitude,nhdplusid,wbd_huc12,wbd_huc12_name,reachcode,gnis_name,catchment_huc12,area_sqkm,length_km,navigable,headwater,coastal,tidal,alaskan
0,IA0000035,001,NPD,NPDES Individual Permit,41.01611,-92.41499,4995479,071000090709,Kettle Creek-Des Moines River,07100009000112,Des Moines River,071000090709,6.2667,2.219704,True,False,False,False,False
1,IA0000051,002,NPD,NPDES Individual Permit,42.57374,-90.69448,13325396,070600030604,Lower Little Maquoketa River,07060003000267,Little Maquoketa River,070600030604,0.855,1.602073,True,False,False,False,False
2,IA0000051,003,NPD,NPDES Individual Permit,42.5739,-90.69423,13325396,070600030604,Lower Little Maquoketa River,07060003000267,Little Maquoketa River,070600030604,0.855,1.602073,True,False,False,False,False
3,IA0000051,005,NPD,NPDES Individual Permit,42.56707,-90.68176,13325578,070600030708,Sinnipee Creek-Mississippi River,07060003000002,Mississippi River,070600030708,6.2343,2.77445,True,False,False,False,False
4,IA0000051,006,NPD,NPDES Individual Permit,42.56698,-90.68171,13325578,070600030708,Sinnipee Creek-Mississippi River,07060003000002,Mississippi River,070600030708,6.2343,2.77445,True,False,False,False,False
